In [1]:
import sys,os,glob
sys.path.append(r'D:\python\neuron-vis\neuronVis')
import numpy as np
import pandas as pd
import BoundLaplace
import json
from PIL import Image
import IONData as IONData
import nrrd
import Flatmap

In [2]:
from Flatmap import map2Flatmap
import joblib

In [3]:
flatenPara = joblib.load(r'D:\python\neuron-vis\resource\flatenPara.pkl')

In [4]:
import os
import nrrd

# 定义本地资源目录（使用原始字符串 r"..." 避免 Windows 路径中的反斜杠转义问题）
resource_dir = r"D:\python\neuron-vis\resource"

# 直接从本地路径读取各 NRRD 文件
grid, header = nrrd.read(os.path.join(resource_dir, "boundlaplace20.nrrd"))
relaxation, relaxationheader = nrrd.read(os.path.join(resource_dir, "boundlaplaceout20.nrrd"))

dv0, dv0header = nrrd.read(os.path.join(resource_dir, "dv0.nrrd"))
dv1, dv1header = nrrd.read(os.path.join(resource_dir, "dv1.nrrd"))
dv2, dv2header = nrrd.read(os.path.join(resource_dir, "dv2.nrrd"))

In [5]:
os.chdir(r"J:\BLA_four_types\csv_PFC")

In [6]:
files = os.listdir()

In [7]:
files

['251034_001_Vglut1_PFC.csv',
 '251034_003_Vglut1_PFC.csv',
 '251034_004_Vglut1_PFC.csv',
 '251034_010_Vglut1_PFC.csv',
 '251039_020_Vglut1_PFC.csv',
 '251039_021_Vglut1_PFC.csv',
 '251039_029_Vglut1_PFC.csv',
 '251039_033_Vglut1_PFC.csv']

In [ ]:
## get point
os.makedirs("..\\csv_PFC_flat",exist_ok=True)
for file in files:
    try:
        df = pd.read_csv(file,index_col=0)
        df1 = df.loc[df.name_use.isin(["PL", "ORB", "ACA", "ILA", "FRP","MO","AI"])&df.terminal == 1]
        # df1 = df.loc[df.name_use.isin(["ORB"])&df.terminal == 1]
        df2= df1[["AP","DV","ML"]]
        soma_xyz = np.array(df2,dtype=float)
        soma_xyz_new = soma_xyz/20
        # soma_xyz_10 = soma_xyz/10
        point=[]
        for p in soma_xyz_new:
            
            if p[2]>285:
                z=285*2-p[2]
            else:
                z=p[2]
            point.append([p[0],p[1],z])

        dv0=dv0.astype(np.float32)/1000-1
        dv1=dv1.astype(np.float32)/1000-1
        dv2=dv2.astype(np.float32)/1000-1
        pointjson = {}
        out=BoundLaplace.ComputeStreamlines(grid,dv0,dv1,dv2,point)
        index=0
        for p in out:
            p2d = map2Flatmap(flatenPara,np.array(p[1])*2,True)
            # print("point:",index,p[0],p2d)
            pointjson[index]=[p2d[0],p2d[1]]
            index+=1

        dftmp = pd.DataFrame(pointjson).T
        dftmp.columns = ["flat_x","flat_y"]
        dfall = pd.concat([df1.reset_index(drop=True), 
                            dftmp.reset_index(drop=True)], 
                        axis=1)
        dfall.to_csv("..\\csv_PFC_flat\\"+file)
    except:
        print(file)
        


In [17]:
flatenPara

[TrackedArray([[1024,  256,  181],
               [1024,  256,  170],
               [1024,  256,  169],
               ...,
               [ 735,  479,   68],
               [ 735,  223,  101],
               [ 735,  223,  241]], dtype=uint64),
 array([False,  True,  True, ...,  True,  True, False]),
 {470850: 0,
  470849.0: 1,
  331723.0: 1,
  331724.0: 1,
  331725.0: 1,
  472243.0: 1,
  609784.0: 1,
  193951.0: 2,
  193952.0: 2,
  193953.0: 2,
  193954.0: 2,
  472244.0: 2,
  333110.0: 2,
  470848.0: 2,
  748996.0: 2,
  611166.0: 2,
  611167.0: 2,
  331887.0: 2,
  609783.0: 2,
  195339.0: 3,
  55194.0: 3,
  55195.0: 3,
  55196.0: 3,
  609949.0: 3,
  888739.0: 3,
  750379.0: 3,
  750381.0: 3,
  750382.0: 3,
  472240.0: 3,
  333107.0: 3,
  748995.0: 3,
  194120.0: 3,
  194121.0: 3,
  55369.0: 3,
  611164.0: 3,
  471013.0: 3,
  331886.0: 3,
  56560.0: 3,
  609782.0: 3,
  1062660.0: 4,
  1062661.0: 4,
  1062662.0: 4,
  890119.0: 4,
  890121.0: 4,
  890122.0: 4,
  195340.0: 4,
  609948.0:

In [14]:
dv1.shape

(660, 400, 570)

In [16]:
dv2.shape

(660, 400, 570)

In [11]:
point

[[163.359, 195.366, 279.24],
 [156.704, 195.51500000000001, 270.723],
 [153.235, 202.038, 267.549],
 [153.44, 134.857, 278.968],
 [159.404, 195.389, 278.442],
 [134.288, 147.852, 276.55],
 [140.895, 188.047, 282.083],
 [158.901, 192.591, 272.592],
 [157.759, 203.035, 275.529],
 [156.466, 203.08800000000002, 269.583],
 [137.602, 167.66500000000002, 281.045],
 [171.711, 169.094, 273.119],
 [140.834, 121.28299999999999, 280.039],
 [155.24200000000002, 197.899, 278.77],
 [148.398, 126.77799999999999, 274.041],
 [142.317, 171.012, 277.237],
 [127.345, 120.33400000000002, 279.224],
 [137.756, 166.902, 282.344],
 [141.958, 177.63, 271.589],
 [140.89000000000001, 130.701, 280.129],
 [156.486, 195.349, 279.909],
 [147.035, 176.31, 275.668],
 [141.572, 168.159, 265.074],
 [137.719, 165.178, 282.997],
 [142.45, 98.7471, 271.94],
 [162.796, 139.928, 276.213],
 [146.341, 157.805, 262.603],
 [127.079, 105.83900000000001, 269.213],
 [145.691, 117.36100000000002, 279.274],
 [149.979, 180.698, 265.009]

In [9]:
out

[[120.15127216514614,
  [93.989630671978, 125.99663067197801, 209.870630671978],
  [180.0757167067528, 212.0827167067528, 295.9567167067528]],
 [110.09532153660555,
  [93.14043647432328, 131.9514364743233, 207.1594364743233],
  [177.22452050828934, 216.03552050828935, 291.24352050828935]],
 [104.37383238588419,
  [92.97473977565767, 141.77773977565766, 207.28873977565763],
  [169.75151650667192, 218.55451650667192, 284.0655165066719]],
 [68.31111289042829,
  [114.0005605840683, 95.4175605840683, 239.52856058406832],
  [195.58214211702347, 176.99914211702347, 321.1101421170235]],
 [115.12329685087585,
  [92.93753357315063, 128.92253357315064, 211.97553357315064],
  [179.0236196079254, 215.00861960792543, 298.0616196079254]],
 [62.242866821481456,
  [98.35206408548356, 111.91606408548355, 240.61406408548356],
  [183.03674871969224, 196.60074871969223, 325.29874871969224]],
 [83.04828191501363,
  [92.94705208063127, 140.09905208063125, 234.13505208063128],
  [173.0271321129799, 220.179132

In [148]:
dfall

,ID,type,AP,DV,ML,R,parent,area_ID,area_name,name_use,bratch_ID,side,terminal,flat_x,flat_y
0,18046,2,3267.18,3907.320,5584.80,1.000000,18045,556,ILA2/3,ILA,122,left,1,871.999924,334.535704
1,19250,2,3134.08,3910.300,5414.46,1.000000,19249,620,ORBm5,ORB,131,left,1,869.179912,324.000904
2,19318,2,3064.70,4040.760,5350.98,1.000000,19317,620,ORBm5,ORB,132,left,1,869.868932,311.976095
3,19608,2,3068.80,2697.140,5579.36,1.000000,19607,171,PL1,PL,135,left,1,924.482850,397.824640
4,19687,2,3188.08,3907.780,5568.84,1.000000,19686,556,ILA2/3,ILA,137,left,1,877.589223,328.339198
5,19908,2,2685.76,2957.040,5531.00,1.417969,19907,304,PL2/3,PL,141,left,1,927.808933,363.319480
6,20902,2,2817.90,3760.940,5641.66,2.250000,20901,484,ORBm1,ORB,149,left,1,926.213723,309.993780
7,20920,2,3178.02,3851.820,5451.84,1.917969,20919,827,ILA5,ILA,150,left,1,868.319564,331.960005
8,20926,2,3155.18,4060.700,5510.58,1.000000,20925,582,ORBm2/3,ORB,151,left,1,876.631529,315.016898
9,20985,2,3129.32,4061.760,5391.66,0.750000,20984,620,ORBm5,ORB,152,left,1,865.935244,316.474878


In [149]:
import matplotlib.pyplot as plt
import copy

In [156]:
files = glob.glob("..\\csv_PFC_flat\\*")

In [157]:
files

['..\\csv_PFC_flat\\251034_001_Vglut1_PFC.csv',
 '..\\csv_PFC_flat\\251034_003_Vglut1_PFC.csv',
 '..\\csv_PFC_flat\\251034_004_Vglut1_PFC.csv',
 '..\\csv_PFC_flat\\251034_010_Vglut1_PFC.csv',
 '..\\csv_PFC_flat\\251039_020_Vglut1_PFC.csv',
 '..\\csv_PFC_flat\\251039_021_Vglut1_PFC.csv',
 '..\\csv_PFC_flat\\251039_029_Vglut1_PFC.csv',
 '..\\csv_PFC_flat\\251039_033_Vglut1_PFC.csv',
 '..\\csv_PFC_flat\\221058_071_Sst.csv',
 '..\\csv_PFC_flat\\251034_001_Vglut1.csv',
 '..\\csv_PFC_flat\\251034_003_Vglut1.csv',
 '..\\csv_PFC_flat\\251034_004_Vglut1.csv',
 '..\\csv_PFC_flat\\251034_010_Vglut1.csv',
 '..\\csv_PFC_flat\\251039_020_Vglut1.csv',
 '..\\csv_PFC_flat\\251039_021_Vglut1.csv',
 '..\\csv_PFC_flat\\251039_029_Vglut1.csv',
 '..\\csv_PFC_flat\\251039_031_Vglut1.csv',
 '..\\csv_PFC_flat\\251039_033_Vglut1.csv']

In [152]:
dfc

,ID,type,AP,DV,ML,R,parent,area_ID,area_name,name_use,bratch_ID,side,terminal,flat_x,flat_y,flat_x.1,flat_y.1,flat_x.1.1,flat_y.1.1
0,18046,2,3267.18,3907.320,5584.80,1.000000,18045,556,ILA2/3,ILA,122,left,1,871.999924,334.535704,871.999924,334.535704,871.999924,334.535704
1,19250,2,3134.08,3910.300,5414.46,1.000000,19249,620,ORBm5,ORB,131,left,1,869.179912,324.000904,869.179912,324.000904,869.179912,324.000904
2,19318,2,3064.70,4040.760,5350.98,1.000000,19317,620,ORBm5,ORB,132,left,1,869.868932,311.976095,869.868932,311.976095,869.868932,311.976095
3,19608,2,3068.80,2697.140,5579.36,1.000000,19607,171,PL1,PL,135,left,1,924.482850,397.824640,924.482850,397.824640,924.482850,397.824640
4,19687,2,3188.08,3907.780,5568.84,1.000000,19686,556,ILA2/3,ILA,137,left,1,877.589223,328.339198,877.589223,328.339198,877.589223,328.339198
5,19908,2,2685.76,2957.040,5531.00,1.417969,19907,304,PL2/3,PL,141,left,1,927.808933,363.319480,927.808933,363.319480,927.808933,363.319480
6,20902,2,2817.90,3760.940,5641.66,2.250000,20901,484,ORBm1,ORB,149,left,1,926.213723,309.993780,926.213723,309.993780,926.213723,309.993780
7,20920,2,3178.02,3851.820,5451.84,1.917969,20919,827,ILA5,ILA,150,left,1,868.319564,331.960005,868.319564,331.960005,868.319564,331.960005
8,20926,2,3155.18,4060.700,5510.58,1.000000,20925,582,ORBm2/3,ORB,151,left,1,876.631529,315.016898,876.631529,315.016898,876.631529,315.016898
9,20985,2,3129.32,4061.760,5391.66,0.750000,20984,620,ORBm5,ORB,152,left,1,865.935244,316.474878,865.935244,316.474878,865.935244,316.474878


In [94]:
colors_hex = [
    '#B43C28',  # 砖红
    '#D27D1E',  # 暖橙
    '#C8C332',  # 芥末黄
    '#3C9B46',  # 翠绿
    '#32AAB9',  # 青蓝
    '#4B78C8',  # 钢蓝
    '#9150C3',  # 柔紫
    '#C850A0'   # 玫瑰粉
]

In [162]:
dfall = pd.DataFrame()
# for file,c in zip(files,colors_hex):
for file in files:    
    dfc = pd.read_csv(file,index_col=0)
    dft = dfc[["flat_x","flat_y","name_use"]]
    dft["file"] = file.split("\\")[-1][0:-4]
    # dft["color"] = c
    dfall = pd.concat([dfall,dft],axis=0)

C:\Users\zljia\AppData\Local\Temp\ipykernel_22424\1949461049.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dft["file"] = file.split("\\")[-1][0:-4]


In [164]:
dfall.name_use.unique()

array(['ORB', 'PL', 'FRP', 'ACA', 'ILA', 'AI', 'MO'], dtype=object)

In [160]:
files

['..\\csv_PFC_flat\\251034_001_Vglut1_PFC.csv',
 '..\\csv_PFC_flat\\251034_003_Vglut1_PFC.csv',
 '..\\csv_PFC_flat\\251034_004_Vglut1_PFC.csv',
 '..\\csv_PFC_flat\\251034_010_Vglut1_PFC.csv',
 '..\\csv_PFC_flat\\251039_020_Vglut1_PFC.csv',
 '..\\csv_PFC_flat\\251039_021_Vglut1_PFC.csv',
 '..\\csv_PFC_flat\\251039_029_Vglut1_PFC.csv',
 '..\\csv_PFC_flat\\251039_033_Vglut1_PFC.csv',
 '..\\csv_PFC_flat\\221058_071_Sst.csv',
 '..\\csv_PFC_flat\\251034_001_Vglut1.csv',
 '..\\csv_PFC_flat\\251034_003_Vglut1.csv',
 '..\\csv_PFC_flat\\251034_004_Vglut1.csv',
 '..\\csv_PFC_flat\\251034_010_Vglut1.csv',
 '..\\csv_PFC_flat\\251039_020_Vglut1.csv',
 '..\\csv_PFC_flat\\251039_021_Vglut1.csv',
 '..\\csv_PFC_flat\\251039_029_Vglut1.csv',
 '..\\csv_PFC_flat\\251039_031_Vglut1.csv',
 '..\\csv_PFC_flat\\251039_033_Vglut1.csv']

In [110]:
def original(i,j,ksize,img):
    #找到矩阵坐标
    x1=y1=-ksize//2
    x2=y2=ksize+x1
    temp=np.zeros(ksize*ksize)
    count=0
    #处理图像
    for m in range(x1,x2):
        for n in range(y1,y2):
            if i+m<0 or i+m>img.shape[0]-1 or j+n<0 or j+n>img.shape[1]-1:
                temp[count]=img[i,j]
            else:
                temp[count]=img[i+m,j+n]
            count +=1
    return temp

#自定义最大值最小值滤波器
def max_min_function(ksize,img,flag):
    img0=copy.copy(img)
    for i in range(0,img.shape[0]):
        for j in range(2,img.shape[1]):
        
            temp=original(i,j,ksize,img0)
            if flag ==0: #设置flag参数，如果0就检测最大值，如果是1检测最小值
                img[i,j]=np.max(temp)
            elif flag==1:
                img[i,j]=np.min(temp)
    return img

In [143]:
dfall.name_use.unique()

array(['ORB', 'PL', 'FRP', 'ACA', 'ILA'], dtype=object)

In [112]:
import cv2
imag_flat,option = nrrd.read(r"D:\python\neuron-vis\resource\flatmap.nrrd")

imag_flat[imag_flat == 484] = 4840
imag_flat[imag_flat == 448] = 4480
blur=max_min_function(3,imag_flat.T,0)
blur = blur.astype(np.uint8)
canny = cv2.Canny(blur,10,20)
bitwise = cv2.bitwise_not(canny)
# cv2.imwrite(r'J:\BLA_four_types\flat\SSp_soma_flat-7.png',bitwise)
fig = plt.figure(figsize=(16,16),dpi=600)
ax = fig.add_subplot(1,1,1)
ax.imshow(bitwise,cmap = 'gray')
for file_name, group in dfall.groupby('file', sort=False):
    # 取出当前分组对应的颜色
    point_color = group['color'].iloc[0]

    ax.scatter(
        group['flat_x'],
        group['flat_y'],
        c=point_color,
        label=file_name,
        s=15,          # 点的大小，可按需调节 (如 10~30)
        alpha=1,     # 点的透明度
        edgecolors='none'
    )

# 设置图像属性与反转 Y 轴（如果 flatmap 坐标系原点在左上方，取消下一行注释）
# ax.invert_yaxis()

ax.set_xlabel('Flat X', fontsize=12)
ax.set_ylabel('Flat Y', fontsize=12)
ax.set_title('Soma Distribution Flatmap', fontsize=14)
ax.set_aspect('equal', adjustable='box')  # 保持 1:1 坐标比例

# 添加图例（放在图右侧外部，防止遮挡散点）
ax.legend(
    bbox_to_anchor=(1.05, 1),
    loc='upper left',
    frameon=False,
    fontsize=9,
    markerscale=1.5
)

# 去除多余的顶部和右侧边框 (美化样式)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

plt.savefig(r'J:\BLA_four_types\flat\SSp_soma_flat-6.png',dpi=300)

C:\Users\zljia\AppData\Local\Temp\ipykernel_22424\2916662428.py:50: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [165]:
dfall.name_use.unique()

array(['ORB', 'PL', 'FRP', 'ACA', 'ILA', 'AI', 'MO'], dtype=object)

In [167]:
import matplotlib.pyplot as plt
import pandas as pd

# 假设你的 DataFrame 变量名为 df
# 包含列: ['flat_x', 'flat_y', 'name_use', 'file', 'color']

# 1. 过滤无效坐标 (0, 0)
df_valid = dfall[(dfall['flat_x'] > 0) & (dfall['flat_y'] > 0)].copy()

# 2. 定义 5 个脑区专属的高区分度柔和颜色字典 (RGB888 Hex)
target_regions = ['ORB', 'PL', 'FRP', 'ACA', 'ILA',"AI","MO"]
region_colors = {
    'ORB': '#B43C28',  # 砖红
    'PL':  '#3C9B46',  # 翠绿
    'FRP': '#4B78C8',  # 钢蓝
    'ACA': '#D27D1E',  # 暖橙
    'ILA': '#9150C3',   # 柔紫
    'AI': "#1E93D2",  # 暖橙
    'MO': "#63C350"   # 柔紫
}

# 3. 筛选目标脑区数据
df_target = df_valid[df_valid['name_use'].isin(target_regions)]

# 4. 创建画板并绘制
fig, ax = plt.subplots(figsize=(16, 16), dpi=300)

for region in target_regions:
    sub_df = df_target[df_target['name_use'] == region]
    if sub_df.empty:
        continue
    ax.imshow(bitwise,cmap = 'gray')
    ax.scatter(
        sub_df['flat_x'],
        sub_df['flat_y'],
        c=region_colors[region],
        label=region,
        s=18,           # 点大小
        alpha=0.75,     # 透明度（略微透明方便观察区域重叠与过渡）
        edgecolors='none'
    )

# 5. 坐标轴与样式配置
ax.set_xlabel('Flat X', fontsize=12)
ax.set_ylabel('Flat Y', fontsize=12)
ax.set_title('PFC Subregions Distribution on Flatmap', fontsize=14, fontweight='bold')
ax.set_aspect('equal', adjustable='box')  # 保持 1:1 空间比例

# 如果图像坐标系原点在左上（像素坐标），取消下面这行注释：
# ax.invert_yaxis()

# 6. 图例设置（按脑区标注）
ax.legend(
    title='Brain Region',
    bbox_to_anchor=(1.03, 1),
    loc='upper left',
    frameon=False,
    fontsize=10,
    title_fontsize=11,
    markerscale=1.6
)

# 隐藏上方和右侧边框
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()
save_path = os.path.join(output_dir, f'flatmap.png')
plt.savefig(save_path, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {save_path}')

C:\Users\zljia\AppData\Local\Temp\ipykernel_22424\271799212.py:68: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


Saved: J:\BLA_four_types\flat\regions_split\flatmap.png


In [131]:
import os
import matplotlib.pyplot as plt
import pandas as pd

# 假设你的 DataFrame 变量名为 df
# 包含列: ['flat_x', 'flat_y', 'name_use', 'file', 'color']

output_dir = r'J:\BLA_four_types\flat\regions_split'
os.makedirs(output_dir, exist_ok=True)

# 1. 过滤掉无效原点坐标 (0, 0)，避免拉偏视图范围
df_valid = dfall[(dfall['flat_x'] > 0) & (dfall['flat_y'] > 0)].copy()

# 2. 按照 name_use 脑区分组出图
for region_name, df_region in df_valid.groupby('name_use', sort=True):
    
    fig, ax = plt.subplots(figsize=(7, 7), dpi=300)
    ax.imshow(bitwise,cmap = 'gray')
    
    # 在当前脑区内，按 file 分组绘制散点并应用对应的 color
    for file_name, group in df_region.groupby('file', sort=False):
        point_color = group['color'].iloc[0]
        
        ax.scatter(
            group['flat_x'],
            group['flat_y'],
            c=point_color,
            label=file_name,
            s=20,
            alpha=0.85,
            edgecolors='none'
        )
    
    # 图像属性配置
    ax.set_title(f'Flatmap - {region_name}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Flat X', fontsize=11)
    ax.set_ylabel('Flat Y', fontsize=11)
    ax.set_aspect('equal', adjustable='box')  # 保持 1:1 物理比例
    
    # 如果 flatmap 的 Y 轴是从上往下递增的像素坐标，取消下面这行注释：
    # ax.invert_yaxis()
    
    # 图例配置（放在图外侧右边）
    ax.legend(
        bbox_to_anchor=(1.05, 1),
        loc='upper left',
        frameon=False,
        fontsize=8,
        markerscale=1.4
    )
    
    # 隐藏上方和右方多余边框
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # 保存并关闭当前图窗
    save_path = os.path.join(output_dir, f'flatmap_{region_name}.png')
    plt.savefig(save_path, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved: {save_path}')

Saved: J:\BLA_four_types\flat\regions_split\flatmap_ACA.png
Saved: J:\BLA_four_types\flat\regions_split\flatmap_FRP.png
Saved: J:\BLA_four_types\flat\regions_split\flatmap_ILA.png
Saved: J:\BLA_four_types\flat\regions_split\flatmap_ORB.png
Saved: J:\BLA_four_types\flat\regions_split\flatmap_PL.png


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 假设你的 DataFrame 变量名为 df
# 包含列: ['flat_x', 'flat_y', 'file', 'color']

fig, ax = plt.subplots(figsize=(8, 8), dpi=300)

# 按 file 分组绘制散点图
for file_name, group in df.groupby('file', sort=False):
    # 取出当前分组对应的颜色
    point_color = group['color'].iloc[0]

    ax.scatter(
        group['flat_x'],
        group['flat_y'],
        c=point_color,
        label=file_name,
        s=15,          # 点的大小，可按需调节 (如 10~30)
        alpha=0.8,     # 点的透明度
        edgecolors='none'
    )

# 设置图像属性与反转 Y 轴（如果 flatmap 坐标系原点在左上方，取消下一行注释）
# ax.invert_yaxis()

ax.set_xlabel('Flat X', fontsize=12)
ax.set_ylabel('Flat Y', fontsize=12)
ax.set_title('Soma Distribution Flatmap', fontsize=14)
ax.set_aspect('equal', adjustable='box')  # 保持 1:1 坐标比例

# 添加图例（放在图右侧外部，防止遮挡散点）
ax.legend(
    bbox_to_anchor=(1.05, 1),
    loc='upper left',
    frameon=False,
    fontsize=9,
    markerscale=1.5
)

# 去除多余的顶部和右侧边框 (美化样式)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()